# Computational Approaches to Modeling and Optimizing Cancer Treatment

This notebook demonstrates computational methods for:
1. Predicting treatment response using machine learning
2. Optimizing drug dosing schedules
3. Analyzing treatment efficacy through statistical methods
4. Visualizing genomic and clinical data

**Author:** Cancer Research Team  
**Date:** 2024  
**License:** MIT

## Overview

This Jupyter notebook provides a comprehensive pipeline for computational cancer treatment optimization. The workflow includes:

- **Data Generation**: Synthetic cancer treatment data for demonstration
- **Exploratory Data Analysis**: Understanding data distributions and patterns
- **Machine Learning**: Random Forest classifier for treatment response prediction
- **Statistical Analysis**: Comparison tests between treatment groups
- **Optimization**: Pharmacokinetic modeling for drug dosing optimization

## Usage

Run cells sequentially to execute the complete analysis pipeline. All outputs are saved automatically.


In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
from scipy.optimize import minimize
from scipy.stats import ttest_ind, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")


## 1. Data Generation and Loading

We'll start by generating synthetic cancer treatment data for demonstration purposes. In a real-world scenario, you would load your actual dataset here.


In [ ]:
def generate_synthetic_cancer_data(n_samples=200, n_genes=50):
    """
    Generate synthetic genomic and clinical data for cancer treatment analysis.
    
    Parameters:
    -----------
    n_samples : int
        Number of patient samples
    n_genes : int
        Number of genomic features
        
    Returns:
    --------
    tuple : (genomic_data, clinical_data, response_labels)
    """
    np.random.seed(42)
    
    # Generate genomic features (e.g., gene expression, mutation counts)
    genomic_data = pd.DataFrame(
        np.random.randn(n_samples, n_genes),
        columns=[f'Gene_{i+1}' for i in range(n_genes)]
    )
    
    # Generate clinical features
    clinical_data = pd.DataFrame({
        'age': np.random.normal(60, 15, n_samples).astype(int),
        'stage': np.random.choice([1, 2, 3, 4], n_samples),
        'tumor_size': np.random.gamma(2, 5, n_samples),
        'mutational_burden': np.random.poisson(10, n_samples),
        'treatment_group': np.random.choice(['Control', 'Treatment'], n_samples)
    })
    
    # Generate response labels based on a model (higher expression of certain genes favors response)
    response_probability = (
        0.3 +
        0.2 * (genomic_data['Gene_1'] > 0) +
        0.2 * (genomic_data['Gene_2'] > 0) +
        0.15 * (clinical_data['treatment_group'] == 'Treatment') +
        0.1 * (clinical_data['age'] < 65) +
        0.05 * (clinical_data['mutational_burden'] > 8)
    )
    
    response_labels = (np.random.rand(n_samples) < response_probability).astype(int)
    response_labels = pd.Series(response_labels, name='response')
    
    return genomic_data, clinical_data, response_labels

# Generate data
print("Generating synthetic cancer treatment data...")
genomic_data, clinical_data, response_labels = generate_synthetic_cancer_data(
    n_samples=200, 
    n_genes=50
)

print(f"Data generated:")
print(f"  Samples: {len(genomic_data)}")
print(f"  Genomic features: {len(genomic_data.columns)}")
print(f"  Clinical features: {len(clinical_data.columns)}")
print(f"  Response rate: {response_labels.mean():.2%}")


## 2. Exploratory Data Analysis

Let's explore the data to understand its structure and identify key patterns.


In [ ]:
# Combine data for analysis
data_combined = pd.concat([genomic_data, clinical_data], axis=1)
data_combined['response'] = response_labels

# Display basic statistics
print("Clinical Data Summary:")
print(clinical_data.describe())
print("\n" + "="*60)
print("\nResponse Distribution:")
print(response_labels.value_counts())
print(f"\nResponse rate: {response_labels.mean():.2%}")

# Visualize clinical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Age distribution
axes[0, 0].hist(clinical_data['age'], bins=20, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Age Distribution')

# Stage distribution
stage_counts = clinical_data['stage'].value_counts().sort_index()
axes[0, 1].bar(stage_counts.index, stage_counts.values, alpha=0.7)
axes[0, 1].set_xlabel('Cancer Stage')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Stage Distribution')

# Mutational burden distribution
axes[1, 0].hist(clinical_data['mutational_burden'], bins=20, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Mutational Burden')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Mutational Burden Distribution')

# Response by treatment group
response_by_treatment = pd.crosstab(clinical_data['treatment_group'], response_labels, normalize='index')
response_by_treatment.plot(kind='bar', ax=axes[1, 1], rot=0)
axes[1, 1].set_xlabel('Treatment Group')
axes[1, 1].set_ylabel('Proportion')
axes[1, 1].set_title('Response Rate by Treatment Group')
axes[1, 1].legend(['Non-responder', 'Responder'])

plt.tight_layout()
plt.savefig('exploratory_analysis.png', dpi=300, bbox_inches='tight')
plt.show()


## 3. Treatment Response Prediction

We'll build a machine learning model to predict treatment response using genomic and clinical features.


In [ ]:
# Prepare features and target
X = pd.concat([genomic_data, clinical_data.drop('treatment_group', axis=1)], axis=1)
y = response_labels

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for better handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"Features: {len(X_train.columns)}")


In [ ]:
# Train Random Forest Classifier
print("Training Random Forest model...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

# Make predictions
y_train_pred = rf_model.predict(X_train_scaled)
y_test_pred = rf_model.predict(X_test_scaled)
y_test_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate model
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_proba)

print(f"\nModel Performance:")
print(f"  Training Accuracy: {train_accuracy:.3f}")
print(f"  Test Accuracy: {test_accuracy:.3f}")
print(f"  Test ROC-AUC: {test_auc:.3f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=['Non-responder', 'Responder']))


In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 15 Most Important Features for Treatment Response Prediction')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTop 10 Most Important Features:")
print(top_features.head(10))


In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC = {test_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Non-responder', 'Responder'],
            yticklabels=['Non-responder', 'Responder'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()


## 4. Statistical Analysis

Perform statistical comparisons between responders and non-responders.


In [ ]:
# Compare key features between responders and non-responders
responders = data_combined[data_combined['response'] == 1]
non_responders = data_combined[data_combined['response'] == 0]

# Statistical comparison for age
age_stat, age_pval = ttest_ind(responders['age'], non_responders['age'])
print("Age Comparison:")
print(f"  Responders: {responders['age'].mean():.2f} ± {responders['age'].std():.2f}")
print(f"  Non-responders: {non_responders['age'].mean():.2f} ± {non_responders['age'].std():.2f}")
print(f"  t-statistic: {age_stat:.3f}, p-value: {age_pval:.4f}")

# Statistical comparison for mutational burden
mut_stat, mut_pval = mannwhitneyu(responders['mutational_burden'], non_responders['mutational_burden'])
print("\nMutational Burden Comparison (Mann-Whitney U test):")
print(f"  Responders median: {responders['mutational_burden'].median():.2f}")
print(f"  Non-responders median: {non_responders['mutational_burden'].median():.2f}")
print(f"  U-statistic: {mut_stat:.3f}, p-value: {mut_pval:.4f}")

# Visualize differences
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Age distribution
axes[0].hist(non_responders['age'], bins=15, alpha=0.6, label='Non-responders', color='red')
axes[0].hist(responders['age'], bins=15, alpha=0.6, label='Responders', color='green')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'Age Distribution (p={age_pval:.4f})')
axes[0].legend()

# Mutational burden
axes[1].hist(non_responders['mutational_burden'], bins=15, alpha=0.6, label='Non-responders', color='red')
axes[1].hist(responders['mutational_burden'], bins=15, alpha=0.6, label='Responders', color='green')
axes[1].set_xlabel('Mutational Burden')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Mutational Burden Distribution (p={mut_pval:.4f})')
axes[1].legend()

plt.tight_layout()
plt.savefig('statistical_comparison.png', dpi=300, bbox_inches='tight')
plt.show()


## 5. Drug Dosing Optimization

Optimize drug dosing schedules using pharmacokinetic modeling.


In [ ]:
class DrugDosingOptimizer:
    """Optimize drug dosing schedules using pharmacokinetic modeling."""
    
    def __init__(self, half_life, max_dose, target_concentration):
        self.half_life = half_life
        self.max_dose = max_dose
        self.target_concentration = target_concentration
    
    def pk_model(self, dose, time_points, clearance_rate, volume):
        """Simple one-compartment pharmacokinetic model."""
        k_el = np.log(2) / self.half_life
        concentrations = (dose / volume) * np.exp(-k_el * time_points)
        return concentrations
    
    def objective(self, doses, time_points, clearance_rate, volume):
        """Minimize deviation from target concentration."""
        total_concentration = np.zeros_like(time_points)
        
        for i, dose in enumerate(doses):
            dose_times = time_points - (i * 24)
            dose_times = dose_times[dose_times >= 0]
            
            if len(dose_times) > 0:
                concentrations = self.pk_model(dose, dose_times, clearance_rate, volume)
                total_concentration[len(time_points) - len(dose_times):] += concentrations
        
        deviation = np.mean((total_concentration - self.target_concentration) ** 2)
        penalty = np.sum(np.maximum(0, doses - self.max_dose) ** 2) * 100
        
        return deviation + penalty
    
    def optimize(self, n_doses=7, clearance_rate=2.0, volume=50.0):
        """Optimize dosing schedule."""
        time_points = np.arange(0, 7 * 24, 1)
        initial_doses = np.ones(n_doses) * self.max_dose * 0.7
        bounds = [(0, self.max_dose) for _ in range(n_doses)]
        
        result = minimize(
            self.objective,
            initial_doses,
            args=(time_points, clearance_rate, volume),
            method='L-BFGS-B',
            bounds=bounds
        )
        
        return result.x, time_points

# Optimize dosing
optimizer = DrugDosingOptimizer(
    half_life=12.0,  # 12 hours
    max_dose=100.0,  # 100 mg
    target_concentration=5.0  # 5 mg/L
)

optimal_doses, time_points = optimizer.optimize(n_doses=7)

print("Optimized Dosing Schedule (7-day cycle):")
for i, dose in enumerate(optimal_doses):
    print(f"  Day {i+1}: {dose:.2f} mg")
print(f"\nTotal dose per cycle: {np.sum(optimal_doses):.2f} mg")

# Visualize pharmacokinetic profile
total_concentration = np.zeros_like(time_points)
for i, dose in enumerate(optimal_doses):
    dose_times = time_points - (i * 24)
    dose_times = dose_times[dose_times >= 0]
    if len(dose_times) > 0:
        concentrations = optimizer.pk_model(dose, dose_times, 2.0, 50.0)
        total_concentration[len(time_points) - len(dose_times):] += concentrations

plt.figure(figsize=(12, 6))
plt.plot(time_points / 24, total_concentration, linewidth=2, label='Drug Concentration')
plt.axhline(y=optimizer.target_concentration, color='r', linestyle='--', 
            label=f'Target Concentration ({optimizer.target_concentration} mg/L)')
plt.xlabel('Days')
plt.ylabel('Concentration (mg/L)')
plt.title('Optimized Drug Concentration Profile Over 7-Day Cycle')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('dosing_optimization.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Summary and Conclusions

This notebook demonstrated:
1. **Data Exploration**: Understanding the structure and characteristics of cancer treatment data
2. **Predictive Modeling**: Building machine learning models to predict treatment response
3. **Statistical Analysis**: Comparing responders and non-responders using appropriate tests
4. **Dosing Optimization**: Using pharmacokinetic modeling to optimize drug dosing schedules

### Key Findings:
- Machine learning models can effectively predict treatment response using genomic and clinical features
- Statistical comparisons help identify biomarkers associated with treatment success
- Pharmacokinetic optimization can help design effective dosing schedules

### Future Directions:
- Integration of multi-omics data (genomics, transcriptomics, proteomics)
- Real-time monitoring and adaptive treatment strategies
- Validation on larger, independent datasets
- Clinical translation and prospective validation
